# Expression tables — `expression_rna.csv` and `expression_rna_hpa.csv`

Verifies tables 3 and 4 of the 10-table schema. These two are kept structurally separate on disk (different files, different columns) — `docs/plan/CONSTRAINTS.md` #9 forbids merging DepMap RNA and HPA into one expression matrix, since HPA's `nTPM` is TMM-normalized and DepMap's `log2(TPM+1)` is not. This notebook checks that separation held, not just the numbers.

In [ ]:
import pandas as pd

In [ ]:
rna_cols = pd.read_csv("../../data/processed/expression_rna.csv", nrows=0).columns.tolist()
hpa_cols = pd.read_csv("../../data/processed/expression_rna_hpa.csv", nrows=0).columns.tolist()
print("expression_rna columns    :", rna_cols)
print("expression_rna_hpa columns:", hpa_cols)

**Confirmed: two separate files with different column sets** (`log2tpm1` only in table 3; `ntpm`/`log2ntpm1` only in table 4) — no merged expression column exists anywhere in `data/processed/`, matching the design.

### Value ranges
`expression_rna` should look like `log2(TPM+1)` (roughly 0-15); `expression_rna_hpa`'s `ntpm` should look like a raw normalized-TPM value (wide range, right-skewed) and its `log2ntpm1` should be on the same rough 0-15 footing as `log2tpm1` (that's the whole point of the transform).

In [ ]:
sample = pd.read_csv("../../data/processed/expression_rna.csv", usecols=["log2tpm1"], nrows=2_000_000)
sample["log2tpm1"].describe()

In [ ]:
hpa_sample = pd.read_csv("../../data/processed/expression_rna_hpa.csv", nrows=500_000)
print("ntpm:")
print(hpa_sample["ntpm"].describe())
print("\nlog2ntpm1:")
print(hpa_sample["log2ntpm1"].describe())

**Confirmed:** `log2tpm1` sample: mean 3.29, range [0, 15.2] — a normal `log2(TPM+1)` shape. `ntpm` sample: mean 44.5, heavily right-skewed (max 28,675) — expected for a raw-ish normalized count. `log2ntpm1` sample: mean 3.26, range [0, 14.8] — sits on almost exactly the same footing as `log2tpm1`, confirming `normalize.normalize_hpa` did what it was supposed to.

### Spot-check: EGFR and MET
The two genes Phase 2's `eda/correlation_eda.ipynb` used to re-establish the real correlation figure (Spearman +0.684). Confirm both resolve in `expression_rna` with plausible values — a full scan, not a sample, since a single gene is a small fraction of an 80M-row file.

In [ ]:
gene_reference = pd.read_csv("../../data/processed/gene_reference.csv")
egfr_id = gene_reference.loc[gene_reference["symbol"] == "EGFR", "ensembl_id"].iloc[0]
met_id = gene_reference.loc[gene_reference["symbol"] == "MET", "ensembl_id"].iloc[0]
print(f"EGFR -> {egfr_id}, MET -> {met_id}")

egfr_rows, met_rows = [], []
for chunk in pd.read_csv("../../data/processed/expression_rna.csv", chunksize=5_000_000):
    egfr_rows.append(chunk[chunk["ensembl_id"] == egfr_id])
    met_rows.append(chunk[chunk["ensembl_id"] == met_id])
egfr_df = pd.concat(egfr_rows)
met_df = pd.concat(met_rows)
print(f"EGFR: n={len(egfr_df)}, mean={egfr_df['log2tpm1'].mean():.3f}, range=[{egfr_df['log2tpm1'].min():.2f}, {egfr_df['log2tpm1'].max():.2f}]")
print(f"MET : n={len(met_df)}, mean={met_df['log2tpm1'].mean():.3f}, range=[{met_df['log2tpm1'].min():.2f}, {met_df['log2tpm1'].max():.2f}]")

**Confirmed: both resolve cleanly.** `EGFR` — 1,495 rows (one per DepMap-RNA profile, since v1 drops no genes), mean 3.515, range [0, 10.53]. `MET` — 1,495 rows, mean 4.315, range [0, 11.09]. Both plausible `log2(TPM+1)` values, consistent with Phase 2's correlation finding.

### Verdict
Both tables are structurally separate, correctly scaled/transformed, and spot-check clean on named genes. No concerns found.